# Getting Started with Snowflake Cortex ML-Based Functions for Wanderlust Voyages

## Overview

For "Wanderlust Voyages," a premier travel company, leveraging data insights is key to staying competitive and enhancing customer experiences. Data/Business Analysts play a crucial role in producing recommendations based on these insights. This often involves building models to: make forecasts for travel demand, identify long-running trends in travel preferences, and detect abnormalities in booking patterns or operational data. However, the statistical and machine learning knowledge required can be a hurdle.

Snowflake Cortex ML-Based Functions provide a SQL-friendly approach to implement these models. As of your current Snowflake version, functions for time-series based data include:

1.  **Forecasting**: Predict future metrics like passenger numbers for specific destinations or package types. This helps Wanderlust Voyages in resource planning, staffing, and inventory management for travel packages.
2.  **Anomaly Detection**: Identify unusual data points, such as unexpected spikes or drops in bookings for a particular region, which could indicate emerging trends, operational issues, or the impact of external factors.
3.  **Contribution Explorer**: (Not the focus of this lab, but useful for) determining the most significant drivers for metrics like booking value or customer satisfaction.

This lab will also demonstrate how Wanderlust Voyages can use **Sentiment Analysis** and **Text Classification** to understand customer feedback and the nature of external events, enriching the forecasting and anomaly detection processes.

For further details on ML Functions, please refer to the [snowflake documentation](https://docs.snowflake.com/guides-overview-analysis).

### Prerequisites
* Working knowledge of SQL
* A Snowflake account login with appropriate roles to create databases, schemas, tables, stages, tasks, and use Cortex functions.

### What You’ll Learn
* How to use Anomaly Detection & Forecasting ML Functions to create models and predict passenger numbers for Wanderlust Voyages.
* How to perform Sentiment Analysis on customer feedback and external event descriptions using `SNOWFLAKE.CORTEX.SENTIMENT`.
* How to classify customer feedback using `SNOWFLAKE.CORTEX.CLASSIFY_TEXT`.
* How to incorporate external event data (with sentiment) into forecasting models.
* How to use Tasks to retrain models on a regular cadence.
* How to use email notification integration to send reports.

### What You’ll Build
This Quickstart will guide you through using Forecasting and Anomaly Detection ML Functions within the context of "Wanderlust Voyages."
We will:
* Create a forecasting model to predict the number of passengers for specific travel offerings, aiding in operational planning.
* Analyze customer feedback to understand sentiment and categorize comments, helping Wanderlust Voyages identify areas for service improvement.
* Analyze external events and their sentiment to understand their potential impact on travel.
* Build an anomaly detection model to identify unusual patterns in passenger numbers for different travel packages/destinations, which can highlight trending offers or issues needing attention.
* **(Optional)** Scale the forecasting model to multiple travel offerings and incorporate sentiment-analyzed external event data to potentially improve forecast accuracy.
* **(Optional)** Showcase how to schedule model retraining and reporting using Tasks and email notifications.

By the end of this lab, you'll have practical experience applying these functions to solve real-world travel analytics problems, empowering you to bring ML insights into your daily work.

Let's get started!

## Setting Up Data in Snowflake for Wanderlust Voyages

### Overview:
You will use Snowflake Notebook to:
* Create Snowflake objects (i.e., warehouse, database, schema).
* Define and load data for bookings, customer feedback, and external events specific to Wanderlust Voyages. This data will form the basis of our analyses.

In [ ]:
-- Create database and schema as specified for Wanderlust Voyages data
CREATE OR REPLACE DATABASE EMBRACE_AI_TOUR_DB;
CREATE OR REPLACE SCHEMA EMBRACE_AI_TOUR_DB.ANALYTICS;
CREATE OR REPLACE WAREHOUSE QUICKSTART_WH;

In [ ]:
-- Set the context for the session
USE DATABASE EMBRACE_AI_TOUR_DB;
USE SCHEMA ANALYTICS;
USE WAREHOUSE QUICKSTART_WH;

### Step 1: Stage
We'll define an S3 stage to load our data.

In [ ]:
-- Create an external stage pointing to s3, to load Wanderlust Voyages data.

CREATE OR REPLACE STAGE s3load
    COMMENT = 'S3 Stage for Wanderlust Voyages Data Load'
    URL = 's3://sfquickstarts/vhol_tui_embrace_ai/';


--List the contents of the stage to verify access and see available files.
LS @s3load;

### Step 2: Loading Wanderlust Voyages Data

We will now create and load the core tables for Wanderlust Voyages: `EXTERNAL_EVENTS_DATA`, `CUSTOMER_FEEDBACK_DATA`, and `BOOKINGS_DATA`.

In [ ]:
-- Create and load EXTERNAL_EVENTS_DATA table
-- Business Value: This table helps Wanderlust Voyages understand external occurrences (e.g., festivals, conferences, travel advisories, economic news)
-- that might impact travel demand or operations in specific regions. Analyzing this data, especially with sentiment, can improve forecasting and risk management.
CREATE OR REPLACE TABLE EMBRACE_AI_TOUR_DB.ANALYTICS.EXTERNAL_EVENTS_DATA (
    EVENT_ID VARCHAR,
    EVENT_DATE DATE,
    EVENT_TYPE VARCHAR,
    IMPACTED_REGION VARCHAR,
    EVENT_DESCRIPTION VARCHAR
);


COPY INTO EMBRACE_AI_TOUR_DB.ANALYTICS.EXTERNAL_EVENTS_DATA
FROM @s3load/customer_feedback/external_event_data.csv.gz;

-- Create and load CUSTOMER_FEEDBACK_DATA table
-- Business Value: This table is crucial for Wanderlust Voyages to gauge customer satisfaction, identify common issues or praises related to bookings,
-- destinations, package types, etc. Sentiment analysis and classification on this data can pinpoint areas for service improvement and product development.
CREATE OR REPLACE TABLE EMBRACE_AI_TOUR_DB.ANALYTICS.CUSTOMER_FEEDBACK_DATA (
    FEEDBACK_ID VARCHAR,
    BOOKING_ID VARCHAR,
    FEEDBACK_DATE DATE,
    RATING INTEGER,
    TOPIC VARCHAR,
    COMMENT_TEXT VARCHAR
);


COPY INTO EMBRACE_AI_TOUR_DB.ANALYTICS.CUSTOMER_FEEDBACK_DATA
FROM @s3load/external_events/customer_feedback_0.csv.gz;

-- Create and load BOOKINGS_DATA table
-- Business Value: This is the core transactional data for Wanderlust Voyages. It tracks all bookings, including details about destination, package type,
-- number of passengers, booking value, and channel. This data is fundamental for sales reporting, revenue analysis, demand forecasting, and understanding customer behavior.
CREATE OR REPLACE TABLE EMBRACE_AI_TOUR_DB.ANALYTICS.BOOKINGS_DATA (
    BOOKING_ID VARCHAR,
    BOOKING_DATE DATE,
    TRAVEL_DATE DATE,
    DESTINATION VARCHAR,
    PACKAGE_TYPE VARCHAR,
    NUMBER_OF_PASSENGERS INTEGER,
    TOTAL_BOOKING_VALUE DECIMAL(10, 2),
    BOOKING_CHANNEL VARCHAR,
    PROMOTION_APPLIED BOOLEAN
);

COPY INTO EMBRACE_AI_TOUR_DB.ANALYTICS.BOOKINGS_DATA
-- This COPY command loads all compatible files from the 'bookings/' directory in the S3 stage.
FROM @s3load/bookings/;

In [ ]:
-- View a sample of the ingested data to verify loading
SELECT * FROM EMBRACE_AI_TOUR_DB.ANALYTICS.BOOKINGS_DATA LIMIT 10;

In [ ]:
-- View a sample of the ingested data to verify loading
SELECT * FROM EMBRACE_AI_TOUR_DB.ANALYTICS.CUSTOMER_FEEDBACK_DATA LIMIT 10;

In [ ]:
-- View a sample of the ingested data to verify loading
SELECT * FROM EMBRACE_AI_TOUR_DB.ANALYTICS.EXTERNAL_EVENTS_DATA LIMIT 10;

## Forecasting Passenger Demand for Wanderlust Voyages

Wanderlust Voyages needs to predict future demand to optimize operations. We'll start by forecasting passenger numbers for a specific offering. Let's assume we're interested in the "City Break Package".

### Step 1: Preparing Data for a Single Travel Offering Forecast

We'll filter the `BOOKINGS_DATA` for our chosen package and aggregate passenger numbers by `TRAVEL_DATE`.

In [ ]:
-- Set search path for ML functions (if not set at account level)
-- This makes calling Cortex functions simpler by adding SNOWFLAKE.ML to the default search path, so we don't have to fully qualify function names like SNOWFLAKE.ML.FORECAST every time.
ALTER SESSION SET SEARCH_PATH = '$current, $public, SNOWFLAKE.ML';

-- Create a table with daily passenger numbers for a specific package type.
-- We'll use TRAVEL_DATE as the timestamp and NUMBER_OF_PASSENGERS as the target.
-- We start by focusing on a single package type, 'City Break', to understand the basics of the FORECAST function before scaling to multiple series.
CREATE OR REPLACE TABLE EMBRACE_AI_TOUR_DB.ANALYTICS.PASSENGER_DEMAND_CITY_BREAK AS (
    SELECT
        TRAVEL_DATE,
        DESTINATION,
        SUM(NUMBER_OF_PASSENGERS) AS TOTAL_PASSENGERS
    FROM
        EMBRACE_AI_TOUR_DB.ANALYTICS.BOOKINGS_DATA
    WHERE
        PACKAGE_TYPE = 'City Break' -- Example package type
        AND TRAVEL_DATE IS NOT NULL
        AND NUMBER_OF_PASSENGERS IS NOT NULL
    GROUP BY
        TRAVEL_DATE, DESTINATION
    ORDER BY
        -- ORDER BY is for reviewing the created table; the FORECAST function handles time series ordering internally based on the TIMESTAMP_COLNAME.
        TRAVEL_DATE
);

-- View the prepared data
SELECT * FROM EMBRACE_AI_TOUR_DB.ANALYTICS.PASSENGER_DEMAND_CITY_BREAK LIMIT 10;

### Step 2: Creating the First Forecasting Model for "City Break" Passengers

We'll use the `FORECAST` ML function to predict future passenger numbers for the "City Break" package in Amsterdam.

This enables proactive planning for logistics, staffing, and marketing efforts related to "City Break" in Amsterdam.

In [ ]:
-- Create a view for the model input
CREATE OR REPLACE VIEW EMBRACE_AI_TOUR_DB.ANALYTICS.AMSTERDAM_CITY_BREAK_PASSENGERS_INPUT_V AS (
    SELECT
        TRAVEL_DATE::timestamp as TRAVEL_DATE,
        sum(TOTAL_PASSENGERS) as TOTAL_PASSENGERS
    FROM
        EMBRACE_AI_TOUR_DB.ANALYTICS.PASSENGER_DEMAND_CITY_BREAK
    WHERE destination = 'Amsterdam'
    GROUP BY 1
);


-- Build Forecasting model for "City Break" passenger numbers.
-- This operation might take a few moments as Snowflake trains the model.
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST AMS_CITY_BREAK_PASSENGER_FORECAST (
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'AMSTERDAM_CITY_BREAK_PASSENGERS_INPUT_V'),
    TIMESTAMP_COLNAME => 'TRAVEL_DATE',
    TARGET_COLNAME => 'TOTAL_PASSENGERS'
);

The forecasting model `AMS_CITY_BREAK_PASSENGER_FORECAST` has been created. We can now use it to generate predictions.

In [ ]:
-- Show models to confirm training has completed
-- This command lists all forecast models you've created in the current schema, allowing you to check their status and details.
SHOW SNOWFLAKE.ML.FORECAST;

### Step 3: Creating and Reviewing Predictions

Let's use our trained `CITY_BREAK_PASSENGER_FORECAST` model to predict passenger numbers for the next 30 days (or a suitable period based on data).

In [ ]:
-- Create predictions for the next 30 periods (days in this case).
CALL AMS_CITY_BREAK_PASSENGER_FORECAST!FORECAST(FORECASTING_PERIODS => 30);

-- Store the results of the forecast into a table for easier querying and reporting.
CREATE OR REPLACE TABLE EMBRACE_AI_TOUR_DB.ANALYTICS.AMS_CITY_BREAK_PREDICTIONS AS (
    SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
);

-- View the predictions
-- Since this is a single-series forecast (total for 'City Break' in Amsterdam), there won't be a 'SERIES' column in the output here.
SELECT * FROM EMBRACE_AI_TOUR_DB.ANALYTICS.AMS_CITY_BREAK_PREDICTIONS ORDER BY TS;

### Step 4: Understanding Forecasting Output & Configuration Options

The prediction output includes:
1.  `TS`: The timestamp for the forecast prediction.
2.  `FORECAST`: The predicted value (total passengers).
3.  `LOWER_BOUND` / `UPPER_BOUND`: The prediction interval, giving a range for the forecast.


The `CONFIG_OBJECT` can be used to adjust parameters like the `prediction_interval`. A value closer to 1 (e.g., 0.95 for 95% interval) gives a wider, more conservative range.
Adjusting the prediction interval allows Wanderlust Voyages to align the forecast's confidence level with their risk appetite. A wider interval might be used for critical planning where underestimation is costly.


In [ ]:
-- Example: Generate forecasts with a 90% prediction interval.
CALL AMS_CITY_BREAK_PASSENGER_FORECAST!FORECAST(
    FORECASTING_PERIODS => 30,
    CONFIG_OBJECT => {'prediction_interval': 0.90}
);

-- Display these results
SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID())) ORDER BY TS;

## Identifying Anomalous Passenger Numbers for Wanderlust Voyages

Beyond forecasting, Wanderlust Voyages wants to identify unusual patterns in passenger numbers for its various offerings. This can help detect emerging trends or unexpected issues.

### Step 1: Building the Anomaly Detection Model

We'll use the `ANOMALY_DETECTION` ML function to identify anomalous passenger numbers for each `City Breaks` in Amsterdam

Detecting anomalies in passenger numbers (e.g., a sudden surge for a specific package-destination or a sharp drop) allows Wanderlust Voyages to investigate the causes. This could uncover successful micro-campaigns, new untapped market segments, negative publicity impacting a destination, or operational disruptions.

In [ ]:
-- Prepare the training data: daily passenger numbers per SERIES_ID.
-- We'll use the MULTI_PACKAGE_DESTINATION_DEMAND_WITH_EVENTS_V view but will not use the exogenous variable for anomaly detection training here,
-- as the ANOMALY_DETECTION function (as of last known details) primarily focuses on the target series itself for unsupervised detection.
-- The series will be the 'SERIES_ID'.
CREATE OR REPLACE VIEW EMBRACE_AI_TOUR_DB.ANALYTICS.AMS_PASSENGER_ANOMALY_DETECTION_INPUT_V AS (
    SELECT
        TRAVEL_DATE AS TIMESTAMP_COL,
        TOTAL_PASSENGERS
    FROM
        EMBRACE_AI_TOUR_DB.ANALYTICS.AMSTERDAM_CITY_BREAK_PASSENGERS_INPUT_V
);

-- Create a training set (e.g., all data except the last month for evaluation)
-- and an analysis set (e.g., the last month of data).
CREATE OR REPLACE VIEW EMBRACE_AI_TOUR_DB.ANALYTICS.PASSENGER_ANOMALY_TRAINING_SET AS (
    SELECT *
    FROM AMS_PASSENGER_ANOMALY_DETECTION_INPUT_V
    -- We are reserving the last month of data to test our anomaly detection model. The training set uses data prior to this.
    WHERE TIMESTAMP_COL < (SELECT MAX(TIMESTAMP_COL) FROM AMS_PASSENGER_ANOMALY_DETECTION_INPUT_V) - INTERVAL '1 MONTH'
);

CREATE OR REPLACE VIEW EMBRACE_AI_TOUR_DB.ANALYTICS.PASSENGER_ANOMALY_ANALYSIS_SET AS (
    SELECT *
    FROM AMS_PASSENGER_ANOMALY_DETECTION_INPUT_V
    -- We are reserving the last month of data to test our anomaly detection model. The training set uses data prior to this.
    WHERE TIMESTAMP_COL >= (SELECT MAX(TIMESTAMP_COL) FROM AMS_PASSENGER_ANOMALY_DETECTION_INPUT_V) - INTERVAL '1 MONTH'
);

In [ ]:
-- Create the Anomaly Detection model. This is an unsupervised model.
-- This might take a few moments.
CREATE OR REPLACE SNOWFLAKE.ML.ANOMALY_DETECTION WANDERLUST_PASSENGER_ANOMALY_MODEL(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'PASSENGER_ANOMALY_TRAINING_SET'),
    TIMESTAMP_COLNAME => 'TIMESTAMP_COL',
    TARGET_COLNAME => 'TOTAL_PASSENGERS',
    -- LABEL_COLNAME is set to an empty string because we are performing unsupervised anomaly detection; we don't have pre-existing labels indicating what is an anomaly.
    LABEL_COLNAME => '');

### Step 2: Detecting Anomalies in Recent Passenger Data

Now, we'll use the trained model to detect anomalies in the most recent data (our `PASSENGER_ANOMALY_ANALYSIS_SET`).

By running this detection regularly, Wanderlust Voyages can flag and investigate unusual passenger numbers promptly for each destination.

This can lead to discovering trending destinations that need more resources or identifying underperforming ones that need marketing adjustments or review.

In [ ]:
-- Call the model to detect anomalies on the analysis set.
CALL WANDERLUST_PASSENGER_ANOMALY_MODEL!DETECT_ANOMALIES(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'PASSENGER_ANOMALY_ANALYSIS_SET'),
    TIMESTAMP_COLNAME => 'TIMESTAMP_COL',
    TARGET_COLNAME => 'TOTAL_PASSENGERS',
    -- The 'prediction_interval' here determines the sensitivity. A value of 0.95 means that points falling outside the 95% confidence interval around the expected value are flagged as anomalies.
    CONFIG_OBJECT => {'prediction_interval': 0.95}
);

-- Store the results into a table
CREATE OR REPLACE TABLE EMBRACE_AI_TOUR_DB.ANALYTICS.WANDERLUST_PASSENGER_ANOMALIES AS (
    SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
);

-- Review the detected anomalies
SELECT * FROM EMBRACE_AI_TOUR_DB.ANALYTICS.WANDERLUST_PASSENGER_ANOMALIES
WHERE IS_ANOMALY = TRUE
ORDER BY SERIES, TS;

The output includes `IS_ANOMALY` (TRUE/FALSE), `Y` (actual value), `FORECAST` (expected value), `LOWER_BOUND`, and `UPPER_BOUND`. Anomalies are points where the actual value falls outside the predicted interval.


## Analyzing Customer Feedback with Sentiment Analysis and Classification for Wanderlust Voyages

In addition to quantitative forecasting, Wanderlust Voyages can gain valuable insights from qualitative data. We'll now explore how to analyze customer comments to understand what their customers are saying. We'll use Snowflake Cortex functions to analyze the `CUSTOMER_FEEDBACK_DATA`.

### Step 1: Sentiment Analysis of Customer Comments

Let's determine the sentiment (positive, negative, neutral) of customer comments.
 Business Value: Understanding the overall sentiment of customer feedback helps Wanderlust Voyages gauge customer satisfaction.

In [ ]:
-- Analyze sentiment of customer comments
-- The SNOWFLAKE.CORTEX.SENTIMENT function returns a score between -1 (negative) and 1 (positive).
-- We can interpret scores around 0 as neutral, >0.2 as positive, and <-0.2 as negative for simplicity.
-- The thresholds (0.2, -0.2) for categorizing sentiment are illustrative. In a real application, these might be tuned based on analyzing a sample of scores and corresponding texts to align with business interpretation.
SELECT
    FEEDBACK_ID,
    RATING,
    COMMENT_TEXT,
    SNOWFLAKE.CORTEX.SENTIMENT(COMMENT_TEXT) AS SENTIMENT_SCORE
FROM
    EMBRACE_AI_TOUR_DB.ANALYTICS.CUSTOMER_FEEDBACK_DATA
WHERE
    COMMENT_TEXT IS NOT NULL AND COMMENT_TEXT != ''
LIMIT 100;

-- Example of categorizing sentiment
SELECT
    FEEDBACK_ID,
    RATING,
    COMMENT_TEXT,
    SNOWFLAKE.CORTEX.SENTIMENT(COMMENT_TEXT) AS SENTIMENT_SCORE,
    CASE
        WHEN SENTIMENT_SCORE > 0.2 THEN 'Positive'
        WHEN SENTIMENT_SCORE < -0.2 THEN 'Negative'
        ELSE 'Neutral'
    END AS SENTIMENT_CATEGORY
FROM
    EMBRACE_AI_TOUR_DB.ANALYTICS.CUSTOMER_FEEDBACK_DATA
WHERE
    COMMENT_TEXT IS NOT NULL AND COMMENT_TEXT != ''
LIMIT 100;

### Step 2: Classifying Customer Feedback Topics

Now, let's try to classify the comments into predefined categories relevant to Wanderlust Voyages' business.

In [ ]:
-- Classify customer comments into relevant topics.
-- We'll define a few example categories. 
-- The list of labels provided to CLASSIFY_TEXT should be tailored to the specific topics Wanderlust Voyages is interested in tracking. These example labels cover common travel-related themes.
SELECT
    FEEDBACK_ID,
    RATING,
    TOPIC AS PREDEFINED_TOPIC, -- Original topic if available
    COMMENT_TEXT,
    SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        COMMENT_TEXT,
        ['Accommodation', 'Tour Quality', 'Customer Service', 'Price', 'Booking Process', 'Destination Experience', 'Flight/Transport']
    ) AS CLASSIFIED_TOPIC
FROM
    EMBRACE_AI_TOUR_DB.ANALYTICS.CUSTOMER_FEEDBACK_DATA
WHERE
    COMMENT_TEXT IS NOT NULL AND COMMENT_TEXT != ''
LIMIT 100;


In [ ]:
-- Let's see how many comments fall into each classified topic and their average sentiment
-- Using a TEMPORARY TABLE to store intermediate results with sentiment scores and classified topics. This is efficient as these values are computed once and then queried for aggregation.
CREATE OR REPLACE TEMPORARY TABLE CUSTOMER_FEEDBACK_SENTIMENT_CLASSIFICATION AS
SELECT
    FEEDBACK_ID,
    RATING,
    COMMENT_TEXT,
    SNOWFLAKE.CORTEX.SENTIMENT(COMMENT_TEXT) AS SENTIMENT_SCORE,
    SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        COMMENT_TEXT,
        ['Accommodation', 'Tour Quality', 'Customer Service', 'Price', 'Booking Process', 'Destination Experience', 'Flight/Transport']
    ) AS CLASSIFIED_TOPIC
FROM
    EMBRACE_AI_TOUR_DB.ANALYTICS.CUSTOMER_FEEDBACK_DATA
WHERE
    COMMENT_TEXT IS NOT NULL AND COMMENT_TEXT != '';

-- Business Value: This aggregated view helps Wanderlust Voyages quickly identify which aspects of their service
-- are receiving positive or negative feedback, guiding strategic decisions. For example, if 'Booking Process'
-- consistently has low sentiment scores, it indicates a need for system improvements.
SELECT
    CLASSIFIED_TOPIC,
    COUNT(*) AS NUMBER_OF_COMMENTS,
    AVG(SENTIMENT_SCORE) AS AVERAGE_SENTIMENT,
    AVG(RATING) AS AVERAGE_RATING
FROM
    CUSTOMER_FEEDBACK_SENTIMENT_CLASSIFICATION
GROUP BY
    CLASSIFIED_TOPIC
ORDER BY
    AVERAGE_SENTIMENT ASC;

## Analyzing External Events with Sentiment Analysis for Wanderlust Voyages

External events can significantly influence travel patterns. Wanderlust Voyages can benefit from understanding the nature of these events.

### Step 1: Applying Sentiment Analysis to External Event Descriptions

We will analyze the `EVENT_DESCRIPTION` in the `EXTERNAL_EVENTS_DATA` table to determine if an event is likely to be perceived positively, negatively, or neutrally.

In [ ]:
-- Create a view with sentiment scores for external events
CREATE OR REPLACE VIEW EMBRACE_AI_TOUR_DB.ANALYTICS.EXTERNAL_EVENTS_SENTIMENT AS
SELECT
    EVENT_ID,
    EVENT_DATE,
    EVENT_TYPE,
    IMPACTED_REGION,
    EVENT_DESCRIPTION,
    SNOWFLAKE.CORTEX.SENTIMENT(EVENT_DESCRIPTION) AS SENTIMENT_SCORE,
    CASE
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(EVENT_DESCRIPTION) > 0.2 THEN 'Positive'
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(EVENT_DESCRIPTION) < -0.2 THEN 'Negative'
        ELSE 'Neutral'
    END AS SENTIMENT_CATEGORY
FROM
    EMBRACE_AI_TOUR_DB.ANALYTICS.EXTERNAL_EVENTS_DATA
WHERE
    EVENT_DESCRIPTION IS NOT NULL AND EVENT_DESCRIPTION != '';

-- View some of the results
SELECT * FROM EMBRACE_AI_TOUR_DB.ANALYTICS.EXTERNAL_EVENTS_SENTIMENT LIMIT 20;


In [ ]:
-- Business Value: Understanding the distribution of event sentiments can inform Wanderlust Voyages about the general climate of external factors.
-- For instance, a high number of negative events in a key destination market might trigger proactive communication or contingency planning.
SELECT
    IMPACTED_REGION,
    EVENT_TYPE,
    SENTIMENT_CATEGORY,
    COUNT(*) AS NUMBER_OF_EVENTS
FROM
    EMBRACE_AI_TOUR_DB.ANALYTICS.EXTERNAL_EVENTS_SENTIMENT
GROUP BY
    ALL
ORDER BY
    IMPACTED_REGION, NUMBER_OF_EVENTS desc;


## Building Multiple Forecasts & Adding External Event Information for Wanderlust Voyages

Wanderlust Voyages offers many packages across various destinations. We can build forecasts for multiple series (package-destination combinations) simultaneously. Furthermore, we'll incorporate our sentiment-analyzed `EXTERNAL_EVENTS_SENTIMENT` data, now linked to specific regions (destinations), to see if it improves predictions.

### Step 1: Prepare Data for Multi-Series Forecast with Region-Specific External Event Sentiments

We'll aggregate passenger numbers by `TRAVEL_DATE`, `PACKAGE_TYPE`, and `DESTINATION`. This will then be joined with our `EXTERNAL_EVENTS_SENTIMENT` data where the `EVENT_DATE` matches `TRAVEL_DATE` and `IMPACTED_REGION` matches `DESTINATION`. 

We will continue to only look at the Destination Amsterdam in this Example but will include all Package Types.

Forecasting demand per package type and destination, while considering region-specific external event sentiments, allows Wanderlust Voyages to
tailor resource allocation, marketing, and pricing strategies with much greater precision. 

For example, a positive event in a specific destination might signal an opportunity to promote packages to that destination more heavily.

In [ ]:
-- Create a view for training data, aggregating passengers by TRAVEL_DATE, PACKAGE_TYPE, and DESTINATION,
-- and joining with external event sentiment specific to the destination (IMPACTED_REGION).
-- A SERIES_ID is created by concatenating PACKAGE_TYPE and DESTINATION for the forecast model.

CREATE OR REPLACE VIEW EMBRACE_AI_TOUR_DB.ANALYTICS.MULTI_PACKAGE_DESTINATION_DEMAND_WITH_EVENTS_V AS
WITH DailyPackageDestinationPassengers AS (
    SELECT
        TRAVEL_DATE,
        PACKAGE_TYPE,
        DESTINATION,
        SUM(NUMBER_OF_PASSENGERS) AS TOTAL_PASSENGERS
    FROM
        EMBRACE_AI_TOUR_DB.ANALYTICS.BOOKINGS_DATA
    WHERE TRAVEL_DATE IS NOT NULL
        AND PACKAGE_TYPE IS NOT NULL
        AND DESTINATION IS NOT NULL
        AND NUMBER_OF_PASSENGERS IS NOT NULL
        -- excluding some data points to be able to
    GROUP BY
        TRAVEL_DATE, PACKAGE_TYPE, DESTINATION
),
DailyEventSentimentByRegion AS (
    SELECT
        EVENT_DATE,
        EVENT_TYPE,
        IMPACTED_REGION, -- Now using IMPACTED_REGION for joining
        AVG(SENTIMENT_SCORE) AS AVG_EVENT_SENTIMENT
    FROM
        EMBRACE_AI_TOUR_DB.ANALYTICS.EXTERNAL_EVENTS_SENTIMENT
    GROUP BY
        EVENT_DATE, IMPACTED_REGION, EVENT_TYPE
)
SELECT
    TO_TIMESTAMP_NTZ(dpdp.TRAVEL_DATE) AS TIMESTAMP_COL,
    dpdp.PACKAGE_TYPE,
    dpdp.DESTINATION,
    COALESCE(des.EVENT_TYPE, 'NoEvent') as EVENT_TYPE,
    dpdp.PACKAGE_TYPE || '_' || dpdp.DESTINATION || '-' || COALESCE(des.EVENT_TYPE, 'NoEvent')  AS SERIES_ID, -- Concatenated series identifier
    dpdp.TOTAL_PASSENGERS,
    COALESCE(des.AVG_EVENT_SENTIMENT, 0) AS EVENT_SENTIMENT_SCORE -- Use 0 if no event for that date-region
FROM
    DailyPackageDestinationPassengers dpdp
LEFT JOIN
    DailyEventSentimentByRegion des ON dpdp.TRAVEL_DATE = des.EVENT_DATE AND dpdp.DESTINATION = des.IMPACTED_REGION
WHERE
    dpdp.DESTINATION = 'Amsterdam'
ORDER BY
    SERIES_ID, dpdp.TRAVEL_DATE;

-- View the prepared data
-- SERIES_ID now combines PACKAGE_TYPE, DESTINATION (Amsterdam), and EVENT_TYPE to create distinct series for forecasting."
SELECT * FROM EMBRACE_AI_TOUR_DB.ANALYTICS.MULTI_PACKAGE_DESTINATION_DEMAND_WITH_EVENTS_V
WHERE EVENT_TYPE != 'NoEvent'
LIMIT 20;

In [ ]:
-- The data is split at '2024-12-01' for this lab. In a real scenario, this split point would be chosen based on data availability and the desired forecast horizon."

-- create view for training data set.
CREATE OR REPLACE VIEW EMBRACE_AI_TOUR_DB.ANALYTICS.TRAINING_MULTI_PACKAGE_DESTINATION_DEMAND_WITH_EVENTS_V AS 
    SELECT * FROM EMBRACE_AI_TOUR_DB.ANALYTICS.MULTI_PACKAGE_DESTINATION_DEMAND_WITH_EVENTS_V
    WHERE timestamp_col < '2024-10-01';

-- create view for testing data set.
CREATE OR REPLACE VIEW EMBRACE_AI_TOUR_DB.ANALYTICS.TESTING_MULTI_PACKAGE_DESTINATION_DEMAND_WITH_EVENTS_V AS 
    SELECT * FROM EMBRACE_AI_TOUR_DB.ANALYTICS.MULTI_PACKAGE_DESTINATION_DEMAND_WITH_EVENTS_V
    WHERE timestamp_col > '2024-10-01';

### Step 2: Build Multi-Series Forecast Model for Wanderlust Voyages (Package & Destination Specific)

We'll use the `SERIES_COLNAME` argument to specify our new `SERIES_ID` (Package_Destination) for individual forecasts.
The `EVENT_SENTIMENT_SCORE` will be an exogenous variable.

Columns in INPUT_DATA other than those specified by SERIES_COLNAME, TIMESTAMP_COLNAME, and TARGET_COLNAME (like EVENT_SENTIMENT_SCORE here) are automatically treated as exogenous variables by the SNOWFLAKE.ML.FORECAST function.

This highly granular multi-series model gives Wanderlust Voyages a powerful, scalable way to generate forecasts across its diverse portfolio.

The inclusion of region-specific event sentiment aims to make these forecasts more reactive and accurate for each specific market.

In [ ]:
-- Train the Multi-Series Forecasting model using SERIES_ID. This might take some time.
-- The EVENT_SENTIMENT_SCORE column from the input view will be automatically used as an exogenous feature.
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST WANDERLUST_MULTI_SERIES_FORECAST (
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'TRAINING_MULTI_PACKAGE_DESTINATION_DEMAND_WITH_EVENTS_V'),
    SERIES_COLNAME => 'SERIES_ID', -- Using the combined series identifier
    TIMESTAMP_COLNAME => 'TIMESTAMP_COL',
    TARGET_COLNAME => 'TOTAL_PASSENGERS'
);

In [ ]:
-- Show models to confirm training has completed
SHOW SNOWFLAKE.ML.FORECAST;

In [ ]:
-- Call the model on the future data to produce predictions:
CALL WANDERLUST_MULTI_SERIES_FORECAST!FORECAST(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'TESTING_MULTI_PACKAGE_DESTINATION_DEMAND_WITH_EVENTS_V'),
    SERIES_COLNAME => 'SERIES_ID',
    TIMESTAMP_COLNAME => 'TIMESTAMP_COL'
);

-- Store results into a table:
CREATE OR REPLACE TABLE EMBRACE_AI_TOUR_DB.ANALYTICS.MULTI_SERIES_PASSENGER_PREDICTIONS AS (
    SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
);

-- View some predictions
SELECT * FROM EMBRACE_AI_TOUR_DB.ANALYTICS.MULTI_SERIES_PASSENGER_PREDICTIONS
ORDER BY SERIES, TS
LIMIT 100;

In [ ]:
-- Compare actual passenger numbers with forecasted numbers
SELECT
    p.SERIES,
    p.TS,
    a.TOTAL_PASSENGERS AS ACTUAL_PASSENGERS,
    p.FORECAST AS FORECASTED_PASSENGERS,
    (a.TOTAL_PASSENGERS - p.FORECAST) AS RESIDUAL,
    p.LOWER_BOUND,
    p.UPPER_BOUND
FROM
    EMBRACE_AI_TOUR_DB.ANALYTICS.MULTI_SERIES_PASSENGER_PREDICTIONS p
JOIN
    EMBRACE_AI_TOUR_DB.ANALYTICS.TESTING_MULTI_PACKAGE_DESTINATION_DEMAND_WITH_EVENTS_V a
    ON p.SERIES = a.SERIES_ID AND p.TS = a.TIMESTAMP_COL
ORDER BY
    p.SERIES, p.TS
LIMIT 100;


### Step 4: Feature Importance & Evaluation Metrics (Package & Destination Specific)

Understanding which factors (including our region-specific `EVENT_SENTIMENT_SCORE`) influence the passenger forecasts for each package-destination series is crucial.

Feature importance helps Wanderlust Voyages understand the unique drivers of demand for each specific package-destination offering.

If event sentiment for a particular region proves to be a significant factor, it validates focused data collection and analysis for that market.

Evaluation metrics per series indicate the reliability of forecasts, guiding confidence in localized business decisions.

In [ ]:
-- Get Feature Importance for the multi-series model
-- EXPLAIN_FEATURE_IMPORTANCE shows which features had the most impact on the model's predictions for each series.

CALL WANDERLUST_MULTI_SERIES_FORECAST!EXPLAIN_FEATURE_IMPORTANCE();
SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

The feature importance output will show the global importance of features like `EVENT_SENTIMENT_SCORE` and auto-generated time features across all series. If `EVENT_SENTIMENT_SCORE` has a reasonable importance, it suggests that external events in specific destinations (as quantified by their sentiment) have a discernible impact on passenger numbers for offerings in those destinations.

Evaluation metrics help assess the model's accuracy for each series.

In [ ]:
-- Evaluate model performance
-- SHOW_EVALUATION_METRICS provides metrics (like MAPE, SMAPE, etc.) for each series in the forecast model, helping assess the prediction accuracy for different package-destination combinations.
CALL WANDERLUST_MULTI_SERIES_FORECAST!SHOW_EVALUATION_METRICS();
SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

### Multi-Series Forecasting with External Events: Summary

In this section, we successfully built a more complex forecasting model that handles multiple series (package-destination combinations) simultaneously and incorporates external event sentiment as an influencing factor. We learned how to:
- Prepare data by joining bookings with external event sentiments.
- Train a multi-series forecast model where Snowflake automatically detects exogenous variables.
- Generate predictions on a test set that includes future values of these exogenous variables.
- Compare actuals vs. forecasted values to assess performance.
- Interpret feature importance and evaluation metrics to understand model drivers and reliability.

**Discussion Points for Wanderlust Voyages:**
* How could the `EVENT_SENTIMENT_SCORE` be further refined? (e.g., different weights for event types?)
* What other exogenous variables could be beneficial? (e.g., holidays, marketing spend)
* How would they act on a series with high forecast error?

## (OPTIONAL) Productionizing Your Workflow for Wanderlust Voyages Using Tasks & Stored Procedures

To keep insights fresh, Wanderlust Voyages can automate model retraining and reporting.

### Step 1: Setting up Tasks for Model Retraining and Anomaly Reporting

We'll create tasks to:
1.  Retrain the anomaly detection model (for `SERIES_ID`) periodically (e.g., monthly).
2.  Detect anomalies using the fresh model.
3.  Send an email report with package-destination combinations showing significant anomalies.

Automating the retraining and reporting process ensures that Wanderlust Voyages decision-makers consistently receive up-to-date information on unusual passenger trends for each package-destination. This allows for agile responses to market dynamics, operational signals, or emerging customer preferences without manual daily/weekly effort from analysts.

In [ ]:
-- Create a task to retrain the anomaly detection model monthly.
-- Note: Ensure the input view/table for training (PASSENGER_ANOMALY_TRAINING_SET logic)
-- is updated to reflect new incoming data before the task runs.
-- This might involve a preceding task that updates the training data.
-- For simplicity, we assume PASSENGER_ANOMALY_TRAINING_SET is appropriately managed.

CREATE OR REPLACE TASK EMBRACE_AI_TOUR_DB.ANALYTICS.AD_PASSENGER_RETRAINING_TASK
    WAREHOUSE = QUICKSTART_WH -- Use your designated warehouse
    -- the task runs at 2:00 AM on the 1st day of every month, in the America/Los_Angeles timezone. Adjust as needed.
    SCHEDULE = 'USING CRON 0 2 1 * * America/Los_Angeles' -- Runs at 2 AM PST on the 1st of every month
AS
CREATE OR REPLACE SNOWFLAKE.ML.ANOMALY_DETECTION WANDERLUST_PASSENGER_ANOMALY_MODEL(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'PASSENGER_ANOMALY_TRAINING_SET'), 
    TIMESTAMP_COLNAME => 'TIMESTAMP_COL',
    TARGET_COLNAME => 'TOTAL_PASSENGERS',
    LABEL_COLNAME => ''
);

-- Stored Procedure to extract anomalies and identify package-destinations with high anomaly counts
-- Stored Procedure to extract anomalies for Amsterdam City Breaks
CREATE OR REPLACE PROCEDURE EMBRACE_AI_TOUR_DB.ANALYTICS.EXTRACT_AMS_CITY_BREAK_ANOMALIES() 
RETURNS TABLE (ANOMALY_TIMESTAMP TIMESTAMP_NTZ, ACTUAL_PASSENGERS NUMBER, EXPECTED_PASSENGERS NUMBER, LOWER_BOUND NUMBER, UPPER_BOUND NUMBER) -- Adjusted return
LANGUAGE SQL
AS
$$
DECLARE
    V_RESULTS CURSOR FOR
        SELECT TS AS ANOMALY_TIMESTAMP, Y AS ACTUAL_PASSENGERS, FORECAST AS EXPECTED_PASSENGERS, "LOWER_BOUND", "UPPER_BOUND"
        FROM EMBRACE_AI_TOUR_DB.ANALYTICS.SP_TEMP_AMS_ANOMALY_RESULTS -- Use consistent temp table name
        WHERE IS_ANOMALY = TRUE
        ORDER BY TS;
BEGIN
    -- Run detection on the latest data for Amsterdam City Breaks
    CALL WANDERLUST_PASSENGER_ANOMALY_MODEL!DETECT_ANOMALIES( -- Model is single-series
        INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'EMBRACE_AI_TOUR_DB.ANALYTICS.PASSENGER_ANOMALY_ANALYSIS_SET'), -- This view is based on AMS data
        TIMESTAMP_COLNAME => 'TIMESTAMP_COL',
        TARGET_COLNAME => 'TOTAL_PASSENGERS',
        CONFIG_OBJECT => {'prediction_interval': 0.95}
        -- CRITICAL: No SERIES_COLNAME
    );

    CREATE OR REPLACE TEMPORARY TABLE EMBRACE_AI_TOUR_DB.ANALYTICS.SP_TEMP_AMS_ANOMALY_RESULTS AS
    SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

    OPEN V_RESULTS;
    RETURN TABLE(RESULTSET_FROM_CURSOR(V_RESULTS));
END;
$$;

### Step 3: Creating a Stored Procedure to Send the Anomaly Report

This procedure will call `EXTRACT_AND_SUMMARIZE_PASSENGER_ANOMALIES` and email the results.
**Replace `<EMAIL-RECIPIENT@example.com>` and `MY_TRAVEL_EMAIL_INT` if you used different names.**

In [ ]:
CREATE OR REPLACE PROCEDURE EMBRACE_AI_TOUR_DB.ANALYTICS.SEND_AMS_CITY_BREAK_ANOMALY_REPORT()
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.9'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'send_email_report'
AS
$$
import snowflake.snowpark as snowpark

def send_email_report(session: snowpark.Session):
    summary_df = session.call('EMBRACE_AI_TOUR_DB.ANALYTICS.EXTRACT_AMS_CITY_BREAK_ANOMALIES') # Call renamed SQL SP

    pandas_df = summary_df.to_pandas() 

    if not pandas_df.empty:
        html_table = pandas_df.to_html(index=False)
        email_subject = "Wanderlust Voyages: Amsterdam City Break Passenger Anomaly Report" # Updated
        email_body_html = f"""
        <html><body>
        <h2>Amsterdam City Break - Passenger Anomaly Report</h2>
        <p>The following anomalies in passenger numbers for Amsterdam City Breaks were detected in the recent period:</p>
        {html_table}
        <p>Please review accordingly.</p>
        </body></html>
        """
        session.call('SYSTEM$SEND_EMAIL', 'MY_TRAVEL_EMAIL_INT', '<EMAIL-RECIPIENT@example.com>', email_subject, email_body_html, 'text/html')
        return "Amsterdam anomaly report sent."
    else:
        email_subject = "Wanderlust Voyages: Amsterdam City Break Anomaly Report - No Anomalies"
        email_body_html = """
        <html><body>
        <h2>Amsterdam City Break - Passenger Anomaly Report</h2>
        <p>No passenger anomalies were detected for Amsterdam City Breaks in the recent period.</p>
        </body></html>
        """
        session.call('SYSTEM$SEND_EMAIL', 'MY_TRAVEL_EMAIL_INT', '<EMAIL-RECIPIENT@example.com>', email_subject, email_body_html, 'text/html')
        return "No significant Amsterdam anomalies to report, email sent."
$$;

### Step 4: Orchestrating the Tasks

Create a task to send the report after the model retraining and anomaly detection.
The `EXTRACT_AND_SUMMARIZE_PASSENGER_ANOMALIES` procedure already runs the detection,
so the `SEND_PASSENGER_ANOMALY_REPORT_TASK` can run after the model retraining.

In [ ]:
CREATE OR REPLACE TASK EMBRACE_AI_TOUR_DB.ANALYTICS.SEND_AMS_ANOMALY_REPORT_TASK 
    WAREHOUSE = QUICKSTART_WH
    AFTER EMBRACE_AI_TOUR_DB.ANALYTICS.AD_AMS_PASSENGER_RETRAINING_TASK 
AS
    CALL EMBRACE_AI_TOUR_DB.ANALYTICS.SEND_AMS_CITY_BREAK_ANOMALY_REPORT(); 

### Step 5: Activating and Testing the Tasks

To run the tasks immediately for testing (after ensuring data is set up):

In [ ]:
-- Resume tasks (they are created in a suspended state)
ALTER TASK EMBRACE_AI_TOUR_DB.ANALYTICS.SEND_AMS_ANOMALY_REPORT_TASK RESUME; -
ALTER TASK EMBRACE_AI_TOUR_DB.ANALYTICS.AD_AMS_PASSENGER_RETRAINING_TASK RESUME; 

-- Manually execute the parent task to trigger the DAG for testing
EXECUTE TASK EMBRACE_AI_TOUR_DB.ANALYTICS.AD_AMS_PASSENGER_RETRAINING_TASK;


## Conclusion for Wanderlust Voyages

**Congratulations!** You've successfully adapted and applied Snowflake Cortex ML-Based Functions to address common analytical challenges for "Wanderlust Voyages."

In this hands-on lab, you've learned how Wanderlust Voyages can:

* **Set up and load core travel data**: Including bookings, customer feedback, and external events.
* **Analyze Customer Feedback**: Use `SNOWFLAKE.CORTEX.SENTIMENT` to understand customer sentiment and `SNOWFLAKE.CORTEX.CLASSIFY_TEXT` to categorize feedback, uncovering areas for service improvement.
* **Assess External Events**: Apply sentiment analysis to external event descriptions to gauge their potential positive or negative impact on travel to specific regions.
* **Forecast Passenger Demand**:
    * Build a forecasting model for passenger numbers of a single travel package.
    * (Optional) Extend this to a multi-series forecast for various package-destination combinations, incorporating the sentiment of region-specific external events as an influencing factor. This helps in granular resource planning and anticipating demand shifts.
* **Detect Anomalies in Passenger Numbers**:
    * Create an anomaly detection model to identify unusual spikes or drops in passenger numbers for different package-destination combinations. This can highlight trending offerings or potential issues requiring investigation at a detailed level.
* **(Optional) Productionize Workflows**:
    * Use Tasks and Stored Procedures to automate model retraining and the generation of email reports on anomalous passenger activity, ensuring timely insights for decision-makers.

By leveraging these Snowflake Cortex capabilities, Wanderlust Voyages can make more data-driven decisions, optimize operations, enhance customer experiences, and ultimately improve its business performance.

### Resources:
For further details on the functions used:
* [Snowflake Cortex ML Functions Overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ml-functions)
* [SNOWFLAKE.ML.FORECAST](https://docs.snowflake.com/en/user-guide/ml-powered-forecasting)
* [SNOWFLAKE.ML.ANOMALY_DETECTION](https://docs.snowflake.com/en/user-guide/ml-powered-anomaly-detection)
* [SNOWFLAKE.CORTEX.SENTIMENT](https://docs.snowflake.com/en/sql-reference/functions/sentiment)
* [SNOWFLAKE.CORTEX.CLASSIFY_TEXT](https://docs.snowflake.com/en/sql-reference/functions/classify)
* [Tasks](https://docs.snowflake.com/en/user-guide/tasks-intro)
* [Stored Procedures](https://docs.snowflake.com/en/developer-guide/stored-procedure/stored-procedures-overview)